# 側抑制の探索: 光学葉エッジ × 未報告モチーフ

2つの問いを **集計を最小化した「1単位=1点」の散布図**で探索する(平均・分率に潰さない)。

1. **エッジ付近の側抑制(1-B)** — 格子境界では surround が物理的に外側で切れる。直接側抑制の
   入力幾何は、各標的細胞の **格子中心へ向かう向き(`inward_unit`)に回転して揃える**と、rim 細胞で
   内向きに偏るか(単なる切断 = 受動的)/偏らずに再分配されるか(能動的補償)が見える。
   *生の hex 空間で rim 細胞をプールすると各細胞の内向き方向がバラバラで非対称が打ち消されるため、
   細胞ごとに inward 基底へ回す。*
2. **未報告の側抑制モチーフ(2-A)** — disynaptic feedforward 抑制 exc(X)→inh(Z)→T を **型三つ組
   (X,Z,T)ごとに1点**で散布(両腕のシナプス数を積に潰さない)。媒介細胞 Z が教科書的な局所介在
   ニューロン科(Dm/Pm/Lai 等)か否かで色分けし、**非正典の Z が強い両腕を持つ外れ値**=報告の薄い
   側抑制媒介の候補をあぶり出す。

限界(全体): 符号は `nt_type`(主に ML 推定)依存、座標は columnar 型のみ(wide-field は home column
を持たないので 1-B では落ちる)。これらは別 Issue の確率的符号・wide-field footprint 解析で補う。

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "src").is_dir() is False:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_DIR
from src.data import FlyWireDataManager
from src.lateral import (
    LOCAL_INTERNEURON_FAMILIES,
    ColumnGeometry,
    LateralInhibitionCriteria,
    SignedConnGraph,
    add_sign,
    assign_stage_from_manager,
    axial_to_cart,
    classify_inhibition,
    hex_distance,
    load_column_assignment,
    output_sign_fraction,
    pq_hemi_maps,
)

EXC_COLOR, INH_COLOR = "tab:red", "tab:blue"  # repo convention: excitation=red, inhibition=blue
HEMI = "right"

m = FlyWireDataManager()
conn = add_sign(m.optic_lobe_connections_df.copy())
col_assign = load_column_assignment(DATA_DIR)
geom = ColumnGeometry.from_assignment(col_assign)

P, Q, HM = pq_hemi_maps(col_assign)
# per-neuron edge geometry, keyed by root_id (joined to the Mi1 reference lattice)
cells = geom.cells.dropna(subset=["region"]).drop_duplicates("root_id").set_index("root_id")
REG = cells["region"].to_dict()
BD = cells["boundary_distance"].to_dict()
IUX = cells["inward_unit_x"].to_dict()
IUY = cells["inward_unit_y"].to_dict()

print(f"edges={len(conn):,}  inhibitory={int((conn['sign']=='inh').sum()):,}  "
      f"column-assigned neurons={len(cells):,}")
print("region counts (Mi1 lattice):")
print(cells["region"].value_counts().to_string())

## 1-B. エッジ変位ベクトル散布 — 直接側抑制の幾何

各 **抑制入力エッジ**(`sign=="inh"`, pre/post とも columnar・同半球)を、pre−post のカラム変位として
Cartesian に直し、**post 細胞の `inward_unit` を +x に揃える回転**を施す。
`inward` 成分 > 0 = ソースが格子中心側、< 0 = 外縁側(rim では多くが格子外)。

In [ ]:
inh = conn[conn["sign"] == "inh"].copy()


def displacement_frame(target):
    """Per-edge inhibitory displacement onto `target` cells, rotated into each post
    cell's inward-aligned frame. One row = one (pre, post) inhibitory edge."""
    s = inh[inh["post_primary_type"] == target].copy()
    for who in ("pre", "post"):
        s[f"{who}_p"] = s[f"{who}_root_id"].map(P)
        s[f"{who}_q"] = s[f"{who}_root_id"].map(Q)
        s[f"{who}_h"] = s[f"{who}_root_id"].map(HM)
    s = s.dropna(subset=["pre_p", "pre_q", "post_p", "post_q"])
    s = s[(s["pre_h"] == HEMI) & (s["post_h"] == HEMI)]
    if s.empty:
        return s
    dp = (s["pre_p"] - s["post_p"]).to_numpy()
    dq = (s["pre_q"] - s["post_q"]).to_numpy()
    dx, dy = axial_to_cart(dp, dq)
    ux = s["post_root_id"].map(IUX).to_numpy()
    uy = s["post_root_id"].map(IUY).to_numpy()
    s["inward"] = dx * ux + dy * uy        # + = toward lattice centre
    s["perp"] = -dx * uy + dy * ux         # signed perpendicular (tangential)
    s["is_lateral"] = (dp != 0) | (dq != 0)  # exclude the home column (co-columnar)
    s["post_region"] = s["post_root_id"].map(REG)
    s["post_bd"] = s["post_root_id"].map(BD)
    return s


# pick well-covered columnar targets: many inhibitory edges with full coordinates
CANDIDATES = ["Mi1", "Mi4", "Mi9", "Tm1", "Tm2", "Tm9", "Tm20", "L1", "L2", "L3", "T4a", "T5a"]
cov = []
for t in CANDIDATES:
    d = displacement_frame(t)
    if len(d):
        cov.append(dict(target=t, edges=len(d),
                        rim=int((d["post_region"] == "rim").sum()),
                        center=int((d["post_region"] == "center").sum())))
cov = pd.DataFrame(cov).sort_values("edges", ascending=False)
print("Coordinate-resolved inhibitory edges per candidate target:")
display(cov)

In [ ]:
TARGET = cov.iloc[0]["target"] if len(cov) else "Mi1"
d = displacement_frame(TARGET)
d = d[d["is_lateral"]]  # lateral (offset) inhibition only; co-columnar removed
regions = ["rim", "middle", "center"]
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
lim = float(np.nanpercentile(np.abs(np.r_[d["inward"], d["perp"]]), 99)) if len(d) else 6
for ax, reg in zip(axes, regions):
    g = d[d["post_region"] == reg]
    ax.axvspan(-lim, 0, color="0.92", zorder=0)  # outward side (largely off-lattice at the rim)
    ax.scatter(g["inward"], g["perp"], s=8 + g["syn_count"], alpha=0.25,
               color=INH_COLOR, edgecolors="none")
    ax.axvline(0, color="k", lw=0.6); ax.axhline(0, color="k", lw=0.6)
    ax.annotate("inward →", (0.62, 0.95), xycoords="axes fraction", fontsize=9)
    ax.annotate("← outward (off-lattice)", (0.02, 0.05), xycoords="axes fraction", fontsize=8, color="0.4")
    ax.set(title=f"{reg}  (n_edges={len(g):,})", xlabel="inward component (column units)")
axes[0].set_ylabel("tangential component")
axes[0].set_xlim(-lim, lim); axes[0].set_ylim(-lim, lim)
plt.suptitle(f"Direct inhibitory inputs onto {TARGET}, rotated into each cell's inward frame "
             f"(shaded = outward/off-lattice side)", y=1.02, fontsize=12)
plt.tight_layout()

**読み方**: center 細胞は原点まわりに等方的な雲になるはず(全方位に隣接カラムがある)。rim 細胞で
点が **inward 側(右半面)に偏り**、shaded な outward 側が空くなら、それは格子の物理的切断を反映する。
重要なのは *inward 側の密度が center と同等か上振れしているか*: 上振れ = 残った内側カラムからの抑制を
増やす能動的再分配、同等 = 単なる受動的切断。次セルで細胞単位に要約する。

### 細胞ごとの内向きバイアス(1標的細胞=1点)
各 post 細胞について、**横方向(home column を除く)** 抑制ソースの `inward` 成分をシナプス数で
重み付け平均し、境界距離に対して散布。0 = 等方、> 0 = 内向き偏り。rim(境界距離小)で系統的に正なら
非対称が実在する。*co-columnar を含めると home column の抑制が重みを支配し非対称が希釈されるため除外。*

In [ ]:
def per_cell_bias(target):
    s = displacement_frame(target)
    s = s[s["is_lateral"]] if len(s) else s  # lateral inhibition only
    if s.empty:
        return pd.DataFrame()
    rows = []
    for rid, c in s.groupby("post_root_id"):
        w = c["syn_count"].to_numpy(dtype=float)
        rows.append(dict(root_id=rid, target=target,
                         mean_inward=float(np.average(c["inward"], weights=w)),
                         n_edges=len(c), bd=BD.get(rid, np.nan), region=REG.get(rid)))
    return pd.DataFrame(rows)


focus_targets = cov["target"].head(4).tolist() if len(cov) else [TARGET]
bias = pd.concat([per_cell_bias(t) for t in focus_targets], ignore_index=True)
bias = bias.dropna(subset=["bd"])

fig, axes = plt.subplots(1, len(focus_targets), figsize=(4.2 * len(focus_targets), 4),
                         sharey=True)
axes = np.atleast_1d(axes)
rcol = {"rim": "tab:orange", "middle": "0.6", "center": "tab:green"}
for ax, t in zip(axes, focus_targets):
    b = bias[bias["target"] == t]
    jit = (np.arange(len(b)) % 7 - 3) * 0.04
    ax.scatter(b["bd"] + jit, b["mean_inward"], s=10, alpha=0.5,
               c=b["region"].map(rcol).fillna("0.8"))
    ax.axhline(0, color="k", lw=0.7)
    ax.set(title=t, xlabel="boundary distance (0=edge)")
axes[0].set_ylabel("syn-weighted mean inward component")
plt.suptitle("Per-cell inward bias of direct inhibition vs distance from the lattice edge "
             "(orange=rim, green=center)", y=1.03, fontsize=11)
plt.tight_layout()
display(bias.groupby(["target", "region"])["mean_inward"].median().unstack().round(3))

## 2-A. 符号付き disynaptic モチーフ散布 — 未報告の媒介細胞をあぶり出す

型レベル符号付きグラフから、**exc(X)→inh(Z)→T** の三つ組を列挙する。`trace_paths` は両腕の重みを積に
するため、ここでは exc エッジと inh エッジを媒介 Z で **直接結合し、両腕のシナプス数を別々に保持**する
(集計は型レベル総和のみ)。表示の都合で両腕に下限 `MIN_ARM` シナプスを課す(明示。silent な切り捨てなし)。

In [ ]:
MIN_ARM = 20
graph = SignedConnGraph(conn)
E = graph.edges
exc = (E[(E["sign"] == "exc") & (E["w"] >= MIN_ARM)][["pre", "post", "w"]]
       .rename(columns={"pre": "X", "post": "Z", "w": "w_xz"}))
inhe = (E[(E["sign"] == "inh") & (E["w"] >= MIN_ARM)][["pre", "post", "w"]]
        .rename(columns={"pre": "Z", "post": "T", "w": "w_zt"}))
motif = exc.merge(inhe, on="Z")
motif = motif[(motif["X"] != motif["Z"]) & (motif["Z"] != motif["T"]) & (motif["X"] != motif["T"])]

vt = m.get_visual_neuron_types_df()[["type", "family"]].drop_duplicates("type")
type2fam = dict(zip(vt["type"], vt["family"]))
motif["Z_family"] = motif["Z"].map(type2fam)
motif["Z_canonical"] = motif["Z_family"].isin(LOCAL_INTERNEURON_FAMILIES)
motif["both_arms"] = motif[["w_xz", "w_zt"]].min(axis=1)
print(f"exc->inh->T triples (both arms >= {MIN_ARM} syn): {len(motif):,}  "
      f"canonical mediator Z: {int(motif['Z_canonical'].sum()):,}  "
      f"non-canonical/novel: {int((~motif['Z_canonical']).sum()):,}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
canon = motif[motif["Z_canonical"]]
novel = motif[~motif["Z_canonical"]]
ax.scatter(canon["w_xz"], canon["w_zt"], s=10, alpha=0.25, color="0.6",
           edgecolors="none", label="canonical mediator Z (Dm/Pm/Lai/...)")
ax.scatter(novel["w_xz"], novel["w_zt"], s=14, alpha=0.55, color=INH_COLOR,
           edgecolors="none", label="non-canonical / novel mediator Z")
ax.set(xscale="log", yscale="log",
       xlabel="X→Z excitation (synapses)", ylabel="Z→T inhibition (synapses)",
       title=f"Disynaptic feedforward-inhibition motifs exc(X)→inh(Z)→T  (1 point = 1 type triple)")
# annotate the strongest novel motifs (both arms large)
for _, r in novel.sort_values("both_arms", ascending=False).head(12).iterrows():
    ax.annotate(f"{r['X']}→{r['Z']}→{r['T']}", (r["w_xz"], r["w_zt"]),
                fontsize=7, alpha=0.9)
ax.legend(loc="lower right")
plt.tight_layout()

**読み方**: 灰=教科書的な局所介在ニューロンが媒介する側抑制(既知)。青=媒介 Z が正典科に属さない三つ組。
**右上の青い外れ値**(両腕とも強い)が、注目すべき非正典の disynaptic 抑制経路。

ただし正直な解釈として、上位の青点は2系統の **大容量・既知** 経路に支配されることが分かる:
(i) **グルタミン酸性の lamina monopolar / transmedullary**(L1, L5, TmY5a 等 — `nt_type=GLUT` ゆえ inh 判定)、
(ii) **遠心性フィードバック**(C2, C3 = medulla→lamina)。これらは介在ニューロンではないが抑制を担う点で
「教科書的な側抑制媒介ではない」ものの、(ii) は段をまたぐ **フィードバック**抑制であり *lateral* ではない。
したがって本散布は「非介在ニューロンによる抑制」を広く surface するが、*真の側抑制* と確定するには媒介 Z が
**同一処理段で空間近傍をプールしている**ことの検証(次段 2-B / Δcolumn surround)が必須。GLUT 媒介の局所
プール(例: 介在科ではない Tm 系)が右上に出れば、それが最も報告の薄い候補になる。

In [ ]:
top_novel = (motif[~motif["Z_canonical"]]
             .sort_values("both_arms", ascending=False)
             .head(25)[["X", "Z", "Z_family", "T", "w_xz", "w_zt", "both_arms"]])
print("Top non-canonical disynaptic inhibition motifs (candidate under-reported mediators):")
display(top_novel.reset_index(drop=True))

## 2-B. 媒介細胞は本当に近傍をプールして段内で抑制するか — 未報告の側抑制媒介

2-A の非正典ヒットは大容量の既知経路(GLUT 性 monopolar・遠心性)に支配されていた。決定打として
媒介候補 Z を **型単位で2つの直交軸**に置く(1型=1点):

- **x = 入力プール半径 `pool_rms`** — Z の興奮性入力ソースのカラム重心まわりの RMS 拡散(列単位)。
  center-surround を作るには近傍をプールする必要がある(columnar relay≈0、wide-field amacrine が大)。
- **y = 段内抑制割合 `same_stage_frac`** — Z の抑制出力のうち同一処理段に向かう割合(段をまたぐ
  feedforward/feedback を除外)。これは座標非依存で「その抑制が *lateral* か」を測る。

点サイズ = その型の右半球細胞数(**網膜部位的にタイル状に並ぶ集団** vs 単一の巨大 tangential を区別)。
色 = 正典の局所介在科か否か。真の側抑制媒介 = *中程度*のプール半径(数カラムの surround)× 高い段内
割合 × タイル状(多細胞)。**非正典でこの帯に入る型 = 報告の薄い側抑制媒介の候補**。

In [ ]:
stage = assign_stage_from_manager(m)
nd = m.optic_lobe_neurons_df
root2type = dict(zip(nd["root_id"], nd["primary_type"]))

# within-stage inhibitory-output fraction per mediator type (coordinate-independent)
cl = classify_inhibition(conn, stage, col_assign=col_assign,
                         criteria=LateralInhibitionCriteria(min_syn=5))
cl["ss_s"] = np.where(cl["same_stage"], cl["syn_count"], 0)
g = cl.groupby("pre_primary_type").agg(
    inh_out=("syn_count", "sum"), ss=("ss_s", "sum"),
    n_target_types=("post_primary_type", "nunique"))
g["same_stage_frac"] = g["ss"] / g["inh_out"]
g = g[g["inh_out"] >= 2000]  # explicit floor: mediators with >=2000 inhibitory output synapses

# input pooling radius: RMS spread (columns) of exc-input source columns about their centroid
cand_ids = set(nd.loc[nd["primary_type"].isin(g.index) & (nd["side"] == "right"), "root_id"])
ex = conn[(conn["sign"] == "exc") & conn["post_root_id"].isin(cand_ids)].copy()
ex["sp"] = ex["pre_root_id"].map(P); ex["sq"] = ex["pre_root_id"].map(Q); ex["sh"] = ex["pre_root_id"].map(HM)
ex = ex.dropna(subset=["sp", "sq"]); ex = ex[ex["sh"] == HEMI]
ex["w"] = ex["syn_count"].astype(float); ex["wp"] = ex["sp"] * ex["w"]; ex["wq"] = ex["sq"] * ex["w"]
cen = ex.groupby("post_root_id").agg(sw=("w", "sum"), swp=("wp", "sum"), swq=("wq", "sum"))
cen["pbar"] = cen["swp"] / cen["sw"]; cen["qbar"] = cen["swq"] / cen["sw"]
ex = ex.join(cen[["pbar", "qbar"]], on="post_root_id")
ex["wd2"] = ex["w"] * hex_distance(ex["sp"] - ex["pbar"], ex["sq"] - ex["qbar"]) ** 2
rr = ex.groupby("post_root_id").agg(swd2=("wd2", "sum"), sw=("w", "sum"), n_src=("w", "size"))
rr = rr[rr["n_src"] >= 3]  # need a few inputs to define a footprint
rr["pool_rms"] = np.sqrt(rr["swd2"] / rr["sw"])
rr["type"] = rr.index.map(root2type)

vt2 = m.get_visual_neuron_types_df()[["type", "family"]].drop_duplicates("type")
fam = dict(zip(vt2["type"], vt2["family"]))
osf = output_sign_fraction(conn, by="pre_primary_type")["inh_frac"]
g = g.join(rr.groupby("type")["pool_rms"].median().rename("pool_rms"))
g = g.join(rr.groupby("type").size().rename("n_cells"))
g["family"] = g.index.map(fam)
g["canonical"] = g["family"].isin(LOCAL_INTERNEURON_FAMILIES)
g["inh_frac_out"] = g.index.map(osf)
g = g.dropna(subset=["pool_rms"])
print(f"mediator types with both axes: {len(g)}  (non-canonical: {int((~g['canonical']).sum())})")

In [ ]:
# local-surround zone: a few-column footprint (not a global tangential field), within-stage, tiled
ZONE = dict(pool_lo=1.5, pool_hi=6.0, ss_min=0.6, ncell_min=20)
fig, ax = plt.subplots(figsize=(11, 7))
ax.axvspan(ZONE["pool_lo"], ZONE["pool_hi"], ymin=ZONE["ss_min"], color="0.93", zorder=0)
canon = g[g["canonical"]]
novel = g[~g["canonical"]]
for sub, col, lab in [(canon, "0.6", "canonical mediator (Dm/Pm/Sm/Li/LPi/LMa/...)"),
                      (novel, INH_COLOR, "non-canonical type")]:
    ax.scatter(sub["pool_rms"], sub["same_stage_frac"],
               s=6 + 3 * np.sqrt(sub["n_cells"]), alpha=0.45, color=col,
               edgecolors="none", label=lab)
cand = novel[(novel["pool_rms"].between(ZONE["pool_lo"], ZONE["pool_hi"]))
             & (novel["same_stage_frac"] >= ZONE["ss_min"]) & (novel["n_cells"] >= ZONE["ncell_min"])]
for t, r in cand.iterrows():
    ax.annotate(t, (r["pool_rms"], r["same_stage_frac"]), fontsize=8, fontweight="bold")
ax.set(xlabel="input pooling radius  (RMS columns of excitatory inputs)",
       ylabel="within-stage fraction of inhibitory output",
       title="Mediator census: who pools a neighborhood AND inhibits within stage?  "
             "(point size ∝ #cells; shaded = local-surround zone)")
ax.annotate("global tangential\nintegrators (few cells)", (12, 0.95), fontsize=8, color="0.4")
ax.annotate("columnar relays", (0.4, 0.2), fontsize=8, color="0.4")
ax.legend(loc="lower right")
plt.tight_layout()

In [ ]:
print("Non-canonical, retinotopically-tiled local lateral-inhibition candidates "
      f"(pool_rms∈[{ZONE['pool_lo']},{ZONE['pool_hi']}], same_stage>={ZONE['ss_min']}, n_cells>={ZONE['ncell_min']}):")
display(cand.sort_values("pool_rms", ascending=False)[
    ["family", "inh_out", "inh_frac_out", "same_stage_frac", "pool_rms", "n_cells", "n_target_types"]].round(2))

**読み方(サニティ)**: columnar relay(Mi1/Mi4)は `pool_rms≈0.7`、遠心性フィードバック C2 は
`same_stage≈0.2`、既知の wide-field amacrine(Dm4/Pm05/LMa5)は wide pool × 高 same_stage —
いずれも期待どおりの位置に来る(手法の妥当性)。**プール半径が極端に大きく細胞数が少ない**点(H1, LT,
LPT 系)は神経節全体を積分する **tangential 大細胞**で、columnar surround ではない(右上の注記)。

**本題の候補**(灰の正典 amacrine 帯に重なる *青*・タイル状): **TmY16・Tm33・MTe01a・LMt1・LC25**。
最も筋が良いのは **TmY16(76細胞)と Tm33(57細胞)** — TmY・Tm は古典的には *興奮性* の feedforward
relay と見なされるが、これらは **抑制出力が優勢(`inh_frac≈1`)・数カラムを近傍プール(pool_rms≈2.7-3.1)・
段内抑制(same_stage≈0.6-0.7)** という局所側抑制媒介の署名を持つ。教科書の Dm/Pm/Sm リストに無いため
**報告が薄い側抑制媒介の候補**。MTe01a(medulla tangential)も同様。なお MeMe 系(medulla-medulla)は
プールは広いが**単一の wide 細胞**(n_cells<20)でタイル状でないため、ここでは columnar surround 候補から
外している(別カテゴリの段内ワイドフィールド)。
確定には (i) 候補 Z が標的の Δcolumn surround を実際に作るか(`disyn_kernel`)、
(ii) `gaba_avg/glut_avg`+`nt_type_score` での符号ロバスト性、(iii) physiology が要る。

## まとめと次の一手

- **1-B(エッジ)**: 直接側抑制の入力を inward 基底に回すと、rim 細胞は系統的な内向きバイアスを示し
  center は示さない(Mi1: center −0.24→rim +0.23、T4a: −0.09→+0.38)— surround が外縁で切れる署名。
  ただし **L2 は全領域で一様に内向き(≈0.48)** で、これは **既知の L4→L2 指向性側抑制**(L4 が支配的
  ソース)をエッジに依らず復元したもの — 手法が本物の指向性側抑制を拾える検証。
- **2-A → 2-B(モチーフ)**: 2-A の生の census は大容量の既知経路(GLUT 性 monopolar・遠心性 C2/C3)に
  支配されて新味が出ない。2-B で「近傍プール × 段内抑制 × タイル状」に絞ると、正典の Dm/Pm/Sm/Li 帯に
  重なる **非正典候補 TmY16・Tm33・MTe01a・LMt1・LC25** が分離する。とくに TmY16・Tm33 は通常 *興奮性
  relay* とみなされる TmY/Tm でありながら抑制性・近傍プール・段内 — **報告の薄い側抑制媒介の候補**。

次の深掘り: (1-C) Mi1/T4a の rim 内向きバイアスを media する具体ソースの surround 復元、
(2-C) 候補 Z(TmY16/Tm33)の `disyn_kernel` で標的の Δcolumn surround を実測、
(2-E) inh→inh→T 脱抑制の横方向版。いずれも符号ロバスト性(`gaba_avg/glut_avg`+`nt_type_score`)の確認込み。